# Notebook 11: Evaluating Agents with LangSmith

**🎯 Goal:** Go beyond basic tracing to learn how to evaluate, test, and score the performance of your agents to ensure they are reliable and effective.

## 🧩 Concept Overview

Once you've built an agent, how do you know if it's any good? How do you prevent it from getting worse when you change a prompt or a tool? The answer is **evaluation**.

- **Tracing** (what we did in Notebook 9) is for debugging **individual runs**. It helps you answer, "What happened in this specific instance?"
- **Evaluation** is for assessing performance **across many runs**. It helps you answer, "How well does my agent perform on a set of representative examples?"

LangSmith provides a complete suite of tools for testing and evaluation. The typical workflow is:
1.  **Create a Dataset:** Collect a set of examples (inputs and optional ground-truth outputs).
2.  **Run your Agent:** Run your agent on the entire dataset.
3.  **Apply Evaluators:** Use automated or human-in-the-loop 'evaluators' to score the agent's responses for things like correctness, relevance, and helpfulness.

## 🖼️ Visual Diagram: The Evaluation Workflow

```
 +-----------------------+
 | Dataset               |
 | - Question 1 + Answer 1 |
 | - Question 2 + Answer 2 |
 | - ...                 |
 +-----------+-----------+
             |
             | Run Test
             v
  +----------+----------+        +---------------------+
  |      Agent v1       | ====>  | Results & Scores v1 |
  | (e.g., Prompt A)    |        | - Correct: 85%      |
  +---------------------+        +----------+----------+
             |                                |
             | Run Test                       | Compare Results
             v                                |
  +----------+----------+        +----------+----------+
  |      Agent v2       | ====>  | Results & Scores v2 |
  | (e.g., Prompt B)    |        | - Correct: 95%      |
  +---------------------+        +---------------------+
```

## ⚙️ Setup & Tracing Review

First, ensure your LangSmith environment variables are set. We'll run a simple agent and look at its trace again, this time focusing on agent-specific details.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain import hub

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Check for LangSmith variables
assert os.getenv("LANGCHAIN_TRACING_V2") == "true", "LANGCHAIN_TRACING_V2 must be 'true'"
assert os.getenv("LANGCHAIN_API_KEY"), "LANGCHAIN_API_KEY must be set"

print("LangSmith is configured correctly.")

### Debugging an Agent with a Trace

Let's create an agent that might fail and see how the trace helps us debug.
Imagine our tool for a calculator is poorly named `run_calculation` but the description is good. The agent might get confused.

In [ ]:
@tool(name="run_calculation")
def calculator(expression: str) -> float:
    """An excellent calculator for evaluating mathematical expressions."""
    return eval(expression)

tools = [calculator]
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input": "What is 42 * 13?"}, config={"run_name": "Agent Debugging Example"})

print("\nAction: Go to the LangSmith trace for 'Agent Debugging Example'. You can see the 'Thought' process where the agent decides to use 'run_calculation' and see if it succeeded or failed.")

## 2. Evaluation 101: Creating a Dataset

The foundation of evaluation is a good dataset. You can create one in code, upload a CSV, or, most conveniently, create one from existing traces.

**Action:**
1.  Go to any run in your LangSmith project (like the one we just created).
2.  In the top-right corner, click the **"Add to Dataset"** button.
3.  Give your new dataset a name (e.g., `agent-test-cases`) and click 'Create'.

That's it! You've just created your first data point for testing.

### Creating a Dataset Programmatically

You can also create datasets in code, which is useful for larger, more structured test cases.

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "My First Programmatic Dataset"

# Check if dataset exists
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name, description="A test dataset for my agent.")
    
    # Add examples (input and optional ground-truth output)
    client.create_examples(
        inputs=[
            {"input": "What is 2+2?"},
            {"input": "What is the capital of France?"}
        ],
        outputs=[
            {"output": "4"},
            {"output": "Paris"}
        ],
        dataset_id=dataset.id
    )
    print(f"Dataset '{dataset_name}' created successfully.")
else:
    print(f"Dataset '{dataset_name}' already exists.")

## 3. Evaluation 102: Running an Evaluator

Once you have a dataset, you can run your agent against it and have LangSmith automatically score the results.

LangChain has many built-in evaluators. A common one is the `"qa"` or `"criteria"` evaluator, which uses an LLM to judge if the agent's output correctly answers the question according to the reference answer.

In [ ]:
from langchain.evaluation import evaluate
from langchain.smith import RunEvalConfig

# The agent we want to test
def agent_to_test(input):
    return agent_executor.invoke(input)

# The evaluation configuration
evaluation_config = RunEvalConfig(
    # We'll use a built-in evaluator that checks for correctness.
    # It uses an LLM to compare the prediction to the reference answer.
    evaluators=["qa"]
)

run = client.run_on_dataset(
    dataset_name=dataset_name,
    llm_or_chain_factory=agent_to_test,
    evaluation=evaluation_config,
    project_name="Agent Evals - Run 1",
    concurrency_level=1 # Run tests one by one
)

print(f"Evaluation run complete! View the results in the '{dataset_name}' dataset in LangSmith.")

## 🔧 Hands-On Exercise

**Goal:** Create a small dataset from agent runs.

1.  Run the `agent_executor` on three different math questions (e.g., "5+5", "100/20", "3^3"). Give each run a unique `run_name` in the `config`.
2.  Go to your LangSmith project.
3.  Find the three runs you just created.
4.  For each run, click **"Add to Dataset"** and add it to a new dataset named `"Math Agent Tests"`.

In [ ]:
agent_executor.invoke({"input": "What is 5+5?"}, config={"run_name": "Math Test 1"})
agent_executor.invoke({"input": "What is 100/20?"}, config={"run_name": "Math Test 2"})
agent_executor.invoke({"input": "What is 3^3?"}, config={"run_name": "Math Test 3"})

print("Three agent runs complete. Go to LangSmith to create your dataset!")

## 🐞 Bug Bounty

The 'bug' here is a **regression**—making a change that accidentally breaks something that used to work. Regressions are very hard to catch without a test dataset.

**Scenario:** Our first calculator tool was perfect. Then, a developer 'improves' the description to be more general, but this makes it more ambiguous and the agent fails.

**Task:** See how running an evaluation test suite would immediately catch this regression.

In [ ]:
# Version 1: Good, specific description
@tool
def calculator_v1(expression: str) -> float:
    """Use this to evaluate math expressions."""
    return eval(expression)
agent_v1 = AgentExecutor(agent=create_react_agent(llm, [calculator_v1], prompt), tools=[calculator_v1])

# Version 2: Bad, ambiguous description (a regression)
@tool
def calculator_v2(expression: str) -> float:
    """Use this for string operations.""" # <-- Bad description!
    return eval(expression)
agent_v2 = AgentExecutor(agent=create_react_agent(llm, [calculator_v2], prompt), tools=[calculator_v2])

print("--- Running V1 (the good one) ---")
agent_v1.invoke({"input": "What is 5 * 5?"})

print("\n--- Running V2 (the regression) ---")
agent_v2.invoke({"input": "What is 5 * 5?"})
print("\nNotice how the second agent fails. If we ran this against our 'Math Agent Tests' dataset, we would instantly see the drop in performance.")

## 💡 Pro Tip

**Build your datasets as you go!** Don't wait until the end of your project. Every time you encounter an interesting edge case, a surprising failure, or a great success in your traces, add it to a dataset. This creates a robust test suite that represents the real-world challenges your agent will face, making your application much more reliable over time.

## 🏁 Summary

You've now learned how to build production-ready agents by systematically evaluating their performance.

1.  **Tracing is for Debugging, Evaluation is for Reliability:** Use traces to inspect single runs and evaluation to measure performance across many runs.
2.  **Datasets are Your Safety Net:** A good dataset of test cases is the best way to catch regressions and ensure that changes to your agent are actual improvements.
3.  **Automated Evaluators Save Time:** LangSmith's built-in evaluators can automatically score your agent's performance on criteria like correctness and helpfulness, allowing you to iterate and improve much faster.